# 05. v10 계좌 피처 19 — 날짜별 마감 스냅샷

팀원 `preprocessing.build_daily_nodes` 와 같은 누적 계산이다. 8/1 부터 하루씩 더하면서 **그날 마감**(당일 거래 포함)까지의
계좌별 19개를 `nodes/<날짜>.npy` (계좌 수 × 19, float32, 행 = `kyc.node_id`) 로 저장한다. 거래가 한 번도 없는 계좌는 0.

| # | 피처 | 정의 (그날 마감까지 누적) |
|---|---|---|
| 1–4 | log_out/in_count, log_out/in_neighbors | 보낸/받은 건수, 고유 상대 수 |
| 5–6 | log_self_count, log_reciprocal_neighbors | 자기 이체 수, 서로 주고받은 상대 수 |
| 7–8 | mean_paid_currency_z, paid_outlier_fraction | 보낸 거래의 금액 z 평균, \|z\|≥3 비율 |
| 9–10 | log_currency_count, log_active_days | 쓴 통화 수, 활동일 수 |
| 11–16 | type_* | 소유주 유형 one-hot |
| 17–19 | log_entity_observed_accounts / out / in_diversity | 같은 소유주의 활동 계좌 수, 소유주 단위 거래 상대 수 |

**주의**: 당일 나중 거래까지 들어간 값이다 (팀원 설계 "day close"). LightGBM 에도 그대로 넣어 GNN 과 조건을 맞춘다.

**디스크**: 계좌 약 212만 × 19 × 4B ≈ 160MB/일. 실험 구간 날짜만 저장하면 84일(08-08~10-30) ≈ 13.5GB (누적은 8/1 부터 매일 한다).

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))                    # code/nonpaten_vs_normal 에서 연다
import importlib, dv_lib as D; importlib.reload(D)

DATASET = "HI-Large"
THREADS, MEMORY = 8, "10GB"        # DuckDB. 컨테이너 한도(약 25GB) 안에서 파이썬 배열과 나눠 쓰도록 절반 이하
P = D.Paths(DATASET)
print(P)

SAVE = "splits"                    # "splits" = 06 이 쓰는 날짜만 / "all" = 91일(08-01~10-30) 전부
CHECK_SQL = True                   # 4절: 스냅샷을 SQL 직접 집계와 비교 (HI-Large 에서 날짜당 수 분)
assert P.edge.exists(), "04 먼저"
work = P.out / "work_nodes.duckdb"
for stale in (work, Path(str(work) + ".wal")):
    stale.unlink(missing_ok=True)
con = D.connect(P, THREADS, MEMORY, db_file=work)
D.register_inputs(con, P)
save_days = None if SAVE == "all" else {d for a, b in D.SPLITS.values() for d in D.days_between(a, b)}
print("저장할 날짜", "전부" if save_days is None else f"{len(save_days)}일")

## 1. 보조표 — 쌍·상호·소유주 쌍의 첫 거래일

- `pair_first` 쌍(A,B)의 첫 거래일 → 그날 A 의 out 이웃, B 의 in 이웃 +1
- `reciprocal_first` A→B 와 B→A 가 둘 다 생긴 날 → 그날 A 의 상호 이웃 +1
- `entity_first` 소유주 쌍의 첫 거래일 → 소유주 out/in 다양성 +1

In [ ]:
t0 = time.time()
D.build_first_days(con, P)
for f in sorted(P.first.glob("*.parquet")):
    print(f"{f.name:24s} {con.execute(f'SELECT count(*) FROM read_parquet({D.q(f)})').fetchone()[0]:>14,} 행")
D.log(f"보조표 {time.time()-t0:,.0f}s")

## 2. 일별 누적 → 스냅샷 저장

하루치 거래만 `edge_features.parquet` 에서 읽어(날짜 정렬이라 그날 row group 만 읽음) numpy bincount 로 누적한다.

In [ ]:
t0 = time.time()
info = D.build_daily_nodes(con, P, save_days=save_days)
D.log(f"전체 {time.time()-t0:,.0f}s")
info

## 3. 확인 — 값 분포, 활동 계좌 수의 증가

In [ ]:
days = sorted(p.stem for p in P.nodes.glob("*.npy"))
print(len(days), "일 저장:", days[0], "~", days[-1])
x = np.load(P.nodes / f"{days[-1]}.npy", mmap_mode="r")
active = (x[:, 9] > 0)
print(f"{days[-1]} 활동 계좌 {active.sum():,} / {len(x):,}")
pd.DataFrame(np.asarray(x[active]), columns=D.NODE_FEATURES).describe().T.round(3)

In [ ]:
# 날짜가 지날수록 누적값이 커지는지 (활동 계좌의 평균 log_in_count / log_active_days)
rows = []
for d in days[::7] + [days[-1]]:
    a = np.load(P.nodes / f"{d}.npy", mmap_mode="r"); m = a[:, 9] > 0
    rows.append(dict(day=d, active=int(m.sum()), log_in_count=float(a[m, 1].mean()), log_active_days=float(a[m, 9].mean())))
pd.DataFrame(rows)

## 4. 확인 — SQL 직접 집계와 비교

같은 날짜까지의 거래를 SQL 로 한 번에 집계한 값(팀원 `build_node_snapshot` 과 같은 식)과 무작위 계좌 2,000개를 비교한다.
누적 계산이 맞으면 차이는 float32 반올림(~1e-6) 수준이어야 한다.

In [ ]:
if CHECK_SQL:
    rng = np.random.default_rng(0)
    for d in [days[0], days[-1]]:
        a = np.load(P.nodes / f"{d}.npy", mmap_mode="r")
        pool = np.flatnonzero(a[:, 9] > 0); ids = np.sort(rng.choice(pool, min(2000, len(pool)), replace=False))
        t0 = time.time(); ref = D.node_snapshot_sql(con, P, d, ids)
        diff = np.abs(np.asarray(a[ids], np.float64) - ref).max(0)
        print(d, f"max|diff| {diff.max():.2e}  ({time.time()-t0:,.0f}s)")
        assert diff.max() < 1e-4, pd.Series(diff, D.NODE_FEATURES)

### (선택) `02_build_features` 의 `features_test/node_features.parquet` 와 비교

그 파일은 10/30 까지 전체 거래로 만든 계좌 프로필이라 **10/30 스냅샷**과 정의가 같다
(금액 z 두 개는 보정표가 달라 제외). 없으면 건너뛴다.

In [ ]:
other = P.base / "features_test" / "node_features.parquet"
if other.exists() and (P.nodes / "2022-10-30.npy").exists():
    a = np.load(P.nodes / "2022-10-30.npy", mmap_mode="r")
    b = pd.read_parquet(other, columns=["node_id"] + D.NODE_FEATURES)
    cols = [c for c in D.NODE_FEATURES if c not in ("mean_paid_currency_z", "paid_outlier_fraction")]
    idx = [D.NODE_FEATURES.index(c) for c in cols]
    diff = np.abs(np.asarray(a[b.node_id.to_numpy()])[:, idx] - b[cols].to_numpy())
    display(pd.Series(diff.max(0), cols, name="max|diff|"))
else:
    print("features_test/node_features.parquet 없음 → 건너뜀")

In [ ]:
D.write_meta(P, "05_nodes", dict(info, saved_days=len(days), created=time.strftime("%Y-%m-%d %H:%M:%S")))
con.close()
for stale in (work, Path(str(work) + ".wal")):
    stale.unlink(missing_ok=True)
print(f"nodes/ {sum(p.stat().st_size for p in P.nodes.glob('*.npy'))/1e9:.1f} GB")